# Plotting in Python with Plotly (`go`, `fig`, and traces)

This notebook teaches the basics of plotting with **Plotly**, focusing on:

- **Line plots** (the “normal” plot)
- **Scatter plots**
- **Bar plots**
- The common workflow using:

```python
fig = go.Figure()
fig.add_trace(go.Scatter(x=x, y=y, mode="lines"))
```

> Why Plotly and not Matplotlib? Every app's `plot_manager.py` in this repo builds its figures with Plotly, since it renders interactive, browser-friendly plots that work well inside Voila -- zoom, pan, and hover all come for free. You'll likely still run into **Matplotlib** (`plt`, `fig`, `ax`) in other people's notebooks or older code: the underlying ideas (a figure containing one or more plots) are the same, but the syntax differs -- e.g. Matplotlib's `axs.plot(x, y)` becomes Plotly's `fig.add_trace(go.Scatter(x=x, y=y, mode="lines"))`.


## 0. Running code cells

- Run a cell with **Shift + Enter**
- If you get an error like `NameError: name 'go' is not defined`, it usually means you forgot to run the import cell below.


In [ ]:
# Run this cell first (Shift+Enter)

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

print("Plotly imported. You're ready to plot!")


## 1. The basic pattern: `fig` and traces

Think of it like this:

- `fig` = the *whole figure* (the canvas, axes, and layout together)
- a **trace** = one plotted series -- a line, a set of markers, a set of bars, ...

You build a figure by creating traces and adding them to `fig`, then styling the whole thing with `fig.update_layout(...)`.


In [ ]:
# A simple dataset
x = np.arange(0, 10)
y = x**2

# Core pattern (single plot)
fig = go.Figure()

fig.add_trace(go.Scatter(x=x, y=y, mode="lines"))   # draw the line
fig.update_layout(
    title="y = x²",
    xaxis_title="x",
    yaxis_title="y",
    width=400,
    height=400,
)

fig.show()


### Common mistake: `width`/`height` are not a tuple

Unlike Matplotlib's `figsize=(4, 4)` tuple, Plotly takes **separate** keyword arguments:

✅ Correct:
```python
fig.update_layout(width=400, height=400)
```

❌ Incorrect:
```python
fig.update_layout(figsize=(400, 400))  # not a Plotly argument
```


## 2. Line plots: multiple lines + legend

Add multiple `go.Scatter(..., mode="lines")` traces to the same figure. Set `name=` on each trace and the legend appears automatically.


In [ ]:
x = np.linspace(0, 4*np.pi, 200)

fig = go.Figure()

## Note that the name (used in the legend) is set per-trace

fig.add_trace(go.Scatter(x=x, y=np.sin(x), mode="lines", name="sin(x)"))
fig.add_trace(go.Scatter(x=x, y=np.cos(x), mode="lines", name="cos(x)"))

fig.update_layout(
    title="Two lines on one plot",
    xaxis_title="x",
    yaxis_title="value",
    width=400,
    height=400,
)

fig.show()


## 3. Scatter plots

Scatter is great when you have (x, y) points and want to see a pattern. Use `mode="markers"`:

```python
fig.add_trace(go.Scatter(x=x, y=y, mode="markers"))
```


In [ ]:
rng = np.random.default_rng(0)

x = rng.normal(loc=0, scale=1, size=60)
y = 0.7 * x + rng.normal(loc=0, scale=0.6, size=60)

fig = go.Figure()

fig.add_trace(go.Scatter(x=x, y=y, mode="markers"))
fig.update_layout(title="Scatter plot", xaxis_title="x", yaxis_title="y", width=400, height=400)
fig.add_hline(y=0, line_color="grey")   # horizontal line at y=0
fig.add_vline(x=0, line_color="grey")   # vertical line at x=0

fig.show()


### Scatter tip: transparency (`opacity`)

If points overlap a lot, add transparency through the marker's `opacity`:


In [ ]:
fig = go.Figure()

fig.add_trace(go.Scatter(x=x, y=y, mode="markers", marker=dict(opacity=0.5)))
fig.update_layout(
    title="Scatter with opacity=0.5", xaxis_title="x", yaxis_title="y", width=400, height=400
)

fig.show()


## 4. Bar plots

Bar plots are useful for **categories** and **counts/values**.

Use:
```python
fig.add_trace(go.Bar(x=categories, y=values))
```


In [ ]:
categories = ["A", "B", "C", "D"]
values = [12, 7, 15, 9]

fig = go.Figure()

fig.add_trace(go.Bar(x=categories, y=values))
fig.update_layout(
    title="Bar plot", xaxis_title="category", yaxis_title="value", width=400, height=400
)

fig.show()


## 5. Subplots: more than one plot in a figure

Use `make_subplots(rows=..., cols=...)` to lay out several plots, then tell each `add_trace` which `row`/`col` it belongs to:

```python
fig = make_subplots(rows=1, cols=2)
fig.add_trace(go.Scatter(...), row=1, col=1)
fig.add_trace(go.Scatter(...), row=1, col=2)
```


In [ ]:
x = np.linspace(0, 100, 100)
y = np.sqrt(x)

fig = make_subplots(rows=1, cols=2, subplot_titles=("Line", "Scatter"))

# Left: line plot
fig.add_trace(go.Scatter(x=x, y=y, mode="lines"), row=1, col=1)

# Right: scatter plot
fig.add_trace(go.Scatter(x=x[::10], y=y[::10], mode="markers"), row=1, col=2)

fig.update_layout(width=800, height=350, showlegend=False)
fig.show()


### Mini rule of thumb

- **One plot** → `fig = go.Figure()` and `fig.add_trace(...)`
- **Multiple plots** → `fig = make_subplots(rows=..., cols=...)` and pass `row=`/`col=` to each `add_trace`


## 6. Subplots grid (2×2)

For a grid, use `rows=2, cols=2` and index each trace with its own `row`/`col`.


In [ ]:
x = np.linspace(0, 2*np.pi, 300)

fig = make_subplots(rows=2, cols=2, subplot_titles=("sin", "cos", "sin vs cos (scatter)", "bar"))

fig.add_trace(go.Scatter(x=x, y=np.sin(x), mode="lines"), row=1, col=1)
fig.add_trace(go.Scatter(x=x, y=np.cos(x), mode="lines"), row=1, col=2)

fig.add_trace(
    go.Scatter(x=np.sin(x[::15]), y=np.cos(x[::15]), mode="markers"), row=2, col=1
)

cats = ["A", "B", "C", "D"]
vals = [3, 8, 5, 11]
fig.add_trace(go.Bar(x=cats, y=vals), row=2, col=2)

fig.update_layout(width=800, height=600, showlegend=False)
fig.show()


## 7. A few useful `fig` methods (you'll use these a lot)

- `fig.update_layout(title="...", xaxis_title="...", yaxis_title="...")`
- `fig.update_xaxes(range=[a, b])`, `fig.update_yaxes(range=[a, b])`
- `fig.update_layout(showlegend=True)` (on automatically once traces have `name=`)
- `fig.add_hline(y=...)`, `fig.add_vline(x=...)`
- `fig.update_xaxes(type="log")`, `fig.update_yaxes(type="log")`


In [ ]:
# Quick demo of x/y ranges (zooming in)
x = np.linspace(0, 10, 300)
y = np.sin(x)

fig = go.Figure()
fig.add_trace(go.Scatter(x=x, y=y, mode="lines"))

fig.update_layout(title="Zoomed view", width=400, height=400)
fig.update_xaxes(range=[2, 8])
fig.update_yaxes(range=[-0.5, 0.5])

fig.show()


## Plotting logarithms


In [ ]:
x = np.linspace(0.0001, 1, 200)
y = np.exp(-1/x)

fig = make_subplots(rows=1, cols=2, subplot_titles=("Linear scale", "Logarithmic y scale"))

# Left: line plot
fig.add_trace(go.Scatter(x=x, y=y, mode="lines"), row=1, col=1)

# Right: same data, log y axis
fig.add_trace(go.Scatter(x=x[::10], y=y[::10], mode="lines"), row=1, col=2)
fig.update_yaxes(type="log", row=1, col=2)

fig.update_layout(width=800, height=350, showlegend=False)
fig.show()


Let's see the case where we have a log-log scale.


In [ ]:
x = np.logspace(-1, 2, 200)
y = np.exp(-1/x)

fig = make_subplots(rows=1, cols=2, subplot_titles=("Linear scale", "log-log scale"))

# Left: line plot
fig.add_trace(go.Scatter(x=x, y=y, mode="lines"), row=1, col=1)

# Right: same data, log-log
fig.add_trace(go.Scatter(x=x[::10], y=y[::10], mode="lines"), row=1, col=2)
fig.update_xaxes(type="log", row=1, col=2)
fig.update_yaxes(type="log", row=1, col=2)

fig.update_layout(width=800, height=350, showlegend=False)
fig.show()


## 8. Saving figures

Plotly figures are interactive by default (zoom, pan, hover), so the most natural way to save one is as **HTML** -- no extra dependency needed:

```python
fig.write_html("my_plot.html")
```

To save a **static image** (PNG, SVG, ...) instead, install the `kaleido` package first (`pip install -q kaleido`), then:

```python
fig.write_image("my_plot.png", scale=2)
```

In Jupyter, the file appears in the same folder as your notebook (unless you give a path).


In [ ]:
# Create and save a figure
x = np.linspace(0, 4, 200)
y = np.exp(-x) * np.cos(6*x)

fig = go.Figure()
fig.add_trace(go.Scatter(x=x, y=y, mode="lines"))
fig.update_layout(title="Saved plot example", width=400, height=400)

fig.write_html("saved_plot_example.html")
fig.show()

print("Saved: saved_plot_example.html")


## 9. Managing the plot style from the beginning (templates)

Say you're preparing plots for a presentation or a manuscript and want a consistent font size and family across all of them. Rather than repeating the same `update_layout(...)` call on every figure, set it once as the default **template** at the top of your notebook:


In [ ]:
import plotly.io as pio

pio.templates["hysprint"] = go.layout.Template(
    layout=go.Layout(
        font=dict(family="Arial", size=14),
        legend=dict(font=dict(size=10)),
        title=dict(font=dict(size=11)),
    )
)
pio.templates.default = "plotly_white+hysprint"


In [ ]:
# Every figure from here on picks up the template automatically
x = np.linspace(0, 4, 200)
y = np.exp(-x) * np.cos(6*x)

fig = go.Figure()
fig.add_trace(go.Scatter(x=x, y=y, mode="lines"))
fig.update_layout(
    title="Styled by the template",
    xaxis_title="range",
    yaxis_title="amplitude",
    width=400,
    height=400,
)

fig.show()


## 10. Exercises (recommended)

Try these without looking back too much:

1. Make a **line plot** of `y = x³` for `x=0..5`.
2. Make a **scatter plot** of 100 random points with `opacity=0.3`.
3. Make a **bar plot** of your top 5 favorite things with scores.
4. Make a **1×3 subplot** with (line, scatter, bar) next to each other.


In [ ]:
# Exercise space 👇

# 1) y = x^3
# x = ...
# y = ...
# fig = go.Figure()
# fig.add_trace(go.Scatter(x=x, y=y, mode="lines"))
# fig.show()

# 2) random scatter (100 points)
# ...

# 3) bar plot (your categories)
# ...

# 4) 1x3 subplots
# ...
